In [56]:
import math, pandas as pd
from math import gcd
from collections import Counter
from scipy.stats import chi2

def primeFactors(n):
    factors = set()
    i=2
    while i * i <= n:
        if n % i == 0:
            while n % i == 0:
                n //= i
        i += 1
    if n > 1:
        factors.add(n)
    return factors

#Test Hull-Dobbell (Vérifie si une séquence pseudo aléatoire est de période maximum)
def isFullPeriod(a, c, m):
    rule1 = gcd(c,m) == 1

    primeM = primeFactors(m)
    rule2 = all((a-1)%p == 0 for p in primeM)

    rule3 = (m % 4 !=0) or ((a-1) % 4 == 0)
    return rule1 and rule2 and rule3

#Formule congruentiel linéaire mixte (Générer la suite pseudo aléatoire)
def xnCompute(a, c , m, x0):
    xn = [x0]
    x1 = ((a * x0) + c) % m
    xn.append(x1)
    for i in range(m-1):
        xn.append(((a * xn[i]) + c) % m)
    return xn

#Test des fréquence
def unCompute(a,c,m,x0):
    xn = xnCompute(a,c,m,x0)
    un = []
    for i in range(len(xn)):
        un.append(xn[i] / m)
    return un

#Fréquence cumulée
def ynCompute(a,c,m,x0):
    un = unCompute(a,c,m,x0)
    yn =[]
    for i in range(len(un)):
        yn.append(int(un[i]*10))
    return yn

#test de saut (Savoir l'espace entre chaque nombre demandé dans la suite)
def jumpTest(a,c,m,x0, studiedNb):
    yn = ynCompute(a,c,m,x0)
    jump = []
    try:
        iPosition = yn.index(studiedNb)
    except ValueError:
        return jump 
    print(iPosition)

    for i in range(iPosition + 1, len(yn)):
        if(yn[i] == studiedNb):
            jump.append(i - iPosition - 1)
            iPosition = i
    return jump

#Test de course (Comparé les nombre 2 a 2 si le premier est supérieur au 2ème = 1 et si 1er supérieur = 2)
def courseTest(a, c, m, x0, size):
    xn = xnCompute(a, c, m, x0)
    course = []
    for i in range(0, size*2, 2):
        if xn[i] > xn[i + 1]:
            course.append(2)
        else:
            course.append(1)
    return course

#Permet de séparer le jeu de données and x groupe d'une taille y
def separating(a, c, m, x0, size, nbGroup):
    yn = ynCompute(a, c, m, x0)
    separated = []
    
    for i in range(0, size*nbGroup, size):
        temp = []
        for i in range(size):
            temp.append(yn[i])
        separated.append(temp)
    return separated

#compte le nombre de chaque combinaison possible dans un test de poker
def pokerCount(a,c,m,x0):
    suite = xnCompute(a,c,m,x0)
    poker = []
    #Dans l'ordre(Poker, Carré, Full, Brelan, Deux pair, une pair, rien)
    pokerCounter = [0,0,0,0,0,0,0]
    for i in range(0,len(suite), 5):
        if i + 5 <= len(suite):
            group= []
            for j in range(5):
                group.append(suite[i+j])
            poker.append(pokerCheck(group))
    for i in range(len(poker)):
        if poker[i] == "Poker":
            pokerCounter[0]+=1
        elif poker[i] == "Carré":
            pokerCounter[1]+=1
        elif poker[i] == "Full":
            pokerCounter[2]+=1
        elif poker[i] == "Brelan":
            pokerCounter[3]+=1
        elif poker[i] == "Deux Pair":
            pokerCounter[4]+=1
        elif poker[i] == "Une Pair":
            pokerCounter[5]+=1
        else:
            pokerCounter[6]+=1
    return pokerCounter

#Poker (dans groupe de 5 vérifie si il y a soit une pair, soit deux pair, soit un brelan(3 les memes) soit un carré(4 les memes) soit un full (une pair + un brelan) soit un poker(5 les memes) soit rien)
def pokerCheck(group):
    count = Counter(group)
    value = count.values()
    if 5 in value:
        return "Poker"
    elif 4 in value:
        return "Carré"
    elif 3 in value and 2 in value:
        return "Full"
    elif 3 in value:
        return "Brelan"
    elif list(value).count(2) ==2:
        return "Deux Pair"
    elif 2 in value:
        return "Une Pair"
    else:
        return "Rien"

#Compte combien de 0 et de 1 sont trouvé dans le test de course
def counting(a, c, m, x0):
    course = courseTest(a, c, m, x0)
    number = []
    one = 0
    two = 0
    for i in range(len(course)):
        if (course[i]) == 1:
            one+=1
        else:
            two+=1
    number.append(one)
    number.append(two)
    return number

#Test du carré-unité (Prendre nombre 4 a 4 pour en faire un graphique)
def carreUnit(a,c,m,x0, size):
    yn = ynCompute(a,c,m,x0)
    carreUnit = []
    for i in range(0, size*4, 4):
        carreUnit.append((yn[i+2] - yn[i])**2 + (yn[i+3] - yn[i+1])**2)
    return carreUnit

#Permet de faire la loi de poisson
def poisson(number):
    un = []
    unCumulated = []
    test = 0
    size = 0
    while(test < 1):
        temp1 = round((math.e** (-number)) * (number**size) / math.factorial(size), 3)
        test += temp1
        un.append(temp1)
        unCumulated.append(round(test, 3))
        size +=1
    return un, unCumulated

#K = inverse d'un de modulo "m" (Permet de trouvé le k pour avoir le x0 a partir de x1)
def kCompute(x1, modulo, a, c):
    k=0
    while((x1 - c + k * modulo) % a != 0):
        k+=1
    return k

def grouping(xi,ri,pi,npi,x):
    i = 0
    while i < len(npi) - 1:
        if npi[i] < 5:
            xi[i] += " + " + xi[i+1]
            ri[i] += ri[i+1]
            pi[i] += pi[i+1]
            npi[i] += npi[i+1]
            x[i] += x[i+1]
            del xi[i+1], ri[i+1], pi[i+1], npi[i+1], x[i+1]
            if i > 0:
                i-=1
        else:
            i+=1
    if len(npi) == 1 and npi[0] <5:
        return False
    return xi, ri, pi, npi, x

def generation(a, c, m, x0):

    full_period = isFullPeriod(a, c, m) 
    suite = xnCompute(a, c, m, x0)

    return(full_period,suite)

def frequence(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : chaque chiffre apparait avec la meme frequence")
    print("H1 : la distribution diffère de l'uniforme")

    print("\nEtape 2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    xn = xnCompute(a, c, m, x0)  

    xi = list(set(xn))
    ri = [xn.count(i) for i in xi]
    pi = [1/len(xi) for i in xi]
    npi = [sum(ri)/len(xi) for i in xi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]
    
    df_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi': pi, 'npi': npi, '(ri-npi)²/npi': i})

    total = pd.DataFrame({'xi': ['Total'], 'ri': [df_tab['ri'].sum()], 'pi': [df_tab['pi'].sum()], 'npi': [df_tab['npi'].sum()], '(ri-npi)²/npi': [df_tab['(ri-npi)²/npi'].sum()]})
    
    df_tab_total = pd.concat([df_tab, total])
    print(df_tab_total.to_string(index=False))

    print("\nEtape 4 :")
    if df_tab['npi'][0] >= 5 :
        print("La condition est respectée pas besoin de regrouper")
    else :
        print("bonne chance")

    print("\nEtape 5 :")
    nb_modalite = df_tab['ri'].sum()
    degre_liberte = nb_modalite - 1
    x2_obs_total = df_tab['(ri-npi)²/npi'].sum()

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)

    print(f"{x2_obs_total:.2f} =< {valeur_critique:.2f}")

    print("\nEtape 6 :")
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : chaque chiffre apparait avec la même fréquence")
    else:
        print("H1 est acceptée : la distribution diffère de l'uniforme")

def poker(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    print("H1 : la distribution diffère de celle attendue")

    print("\nEtape2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    
    xi = ["Poker", "Carré", "Full", "Brelan", "Deux Pair", "Une Pair", "Rien"]
    ri = pokerCount(a,c,m,x0)
    pi = [1/(10**4), 450/(10**5), 900/(10**5), 7200/(10**5), 10800/(10**5), 50400/(10**5), 0.3024]
    npi = [sum(ri)*p for p in pi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]

    dp_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
    print(dp_tab.to_string(index=False))

    print("\nEtape 4 :")
    grouped = grouping(xi,ri,pi,npi,i)
    if(grouped):
        dpg_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
        print(dpg_tab.to_string(index=False))
        nb_modalite = dpg_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dpg_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    else:
        print("Toujours inférieur a 5 aprés regroupement de toutes les catégories")
        nb_modalite = dp_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dp_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    
    print("\nEtape 5 :")

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)
    print("\nv = %f" % valeur_critique)

    print("\nEtape 6 :")
    
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    else:
        print("H1 est acceptée : la distribution diffère de celle attendue")
        
def partie1(a,c,m,x0):
    full_period, suite = generation(a, c, m, x0)

    if not full_period:
        print("Le 3 hypothèses du théorème de Hull-Dobell ne sont pas respectées")
    else:
        print("Les 3 hypothèses du théorème de Hull-Dobell sont respectées")
        print("\nTest de fréquence en six étapes :\n")
        frequence(a,c,m,x0)
        print("\nTest de poker en six étapes :\n")
        poker(a,c,m,x0)
partie1(41,13,100,7)

Les 3 hypothèses du théorème de Hull-Dobell sont respectées

Test de fréquence en six étapes :

Etape 1 :
H0 : chaque chiffre apparait avec la meme frequence
H1 : la distribution diffère de l'uniforme

Etape 2 :
0.05

Etape 3 :
   xi  ri       pi        npi  (ri-npi)²/npi
    0   2 0.019608   1.980392       0.000194
    1   2 0.019608   1.980392       0.000194
    2   2 0.019608   1.980392       0.000194
    3   2 0.019608   1.980392       0.000194
    6   2 0.019608   1.980392       0.000194
    7   1 0.019608   1.980392       0.485343
    8   2 0.019608   1.980392       0.000194
   11   2 0.019608   1.980392       0.000194
   13   2 0.019608   1.980392       0.000194
   19   2 0.019608   1.980392       0.000194
   20   2 0.019608   1.980392       0.000194
   24   2 0.019608   1.980392       0.000194
   25   2 0.019608   1.980392       0.000194
   27   2 0.019608   1.980392       0.000194
   29   2 0.019608   1.980392       0.000194
   30   2 0.019608   1.980392       0.000194
   31  

In [57]:
def partie2(a,c,m,x0):

    poisson_client = poisson(1.5)
    poisson_client_prio = poisson(0.7)
    prio_absolu = 0,3
    
loi_duree_service = pd.DataFrame({
        "Durée en minutes": [1, 2, 3, 4, 5, 6],
        "Répétition": [24, 18, 10, 3, 3, 2]
})

COUTS = {
    "presence_ord": 15 / 60,
    "presence_pr_rel": 35 / 60,
    "presence_pr_abs": 45 / 60,
    "occup_pr": 33 / 60,
    "occup_ord": 28 / 60,
    "inoccup": 18 / 60,
    "perte_pr": 20,
    "perte_ord": 15
}


def calcul_min_station():

        client = 1.5
        client_prio = 0.7

        total_activite = client + client_prio
        somme_répétition = loi_duree_service["Répétition"].sum()
        durée_moyenne_service = sum(loi_duree_service["Durée en minutes"] * loi_duree_service["Répétition"]) / somme_répétition

        calcul_psy = total_activite * durée_moyenne_service

        rounded_value = round(calcul_psy)
        print(rounded_value)




    

    

In [58]:
calcul_min_station()

5


In [59]:
import numpy as np
import pandas as pd

# -------------------------------
# Paramètres
# -------------------------------
np.random.seed(42)  # reproductibilité

lambda_ord = 1.5
lambda_prio = 0.7
prio_absolu_ratio = 0.3

# Loi empirique des durées de service
durations = [1, 2, 3, 4, 5, 6]
weights = [24, 18, 10, 3, 3, 2]
proba_durations = np.array(weights) / sum(weights)

# Coûts
COUTS = {
    "presence_ord": 15 / 60,
    "presence_pr_rel": 35 / 60,
    "presence_pr_abs": 45 / 60,
    "occup_pr": 33 / 60,
    "occup_ord": 28 / 60,
    "inoccup": 18 / 60,
    "perte_pr": 20,
    "perte_ord": 15,
}

# -------------------------------
# Génération des arrivées
# -------------------------------
def generate_arrivals():
    nb_ord = np.random.poisson(lambda_ord)
    nb_prio = np.random.poisson(lambda_prio)
    arrivals = []
    
    # Clients ordinaires
    for _ in range(nb_ord):
        d = np.random.choice(durations, p=proba_durations)
        arrivals.append({"type": "ord", "duration": d})
    
    # Clients prioritaires
    for _ in range(nb_prio):
        d = np.random.choice(durations, p=proba_durations)
        if np.random.rand() < prio_absolu_ratio:
            arrivals.append({"type": "prio_abs", "duration": d})
        else:
            arrivals.append({"type": "prio_rel", "duration": d})
    
    return arrivals

# -------------------------------
# Simulation
# -------------------------------
def simulate(nbStations, tempsSimul=20, verbose=True):
    stations = [None] * nbStations
    file = []
    file_cumulee = 0
    
    for minute in range(1, tempsSimul+1):
        if verbose:
            print(f"\n=== Minute {minute} ===")
            print("Stations (avant):", stations)
            print("File (avant):", file)
        
        # Générer arrivées
        arrivals = generate_arrivals()
        if verbose:
            print("Arrivées:", arrivals)
        
        # Placer dans la file (priorité absolue > relative > ordinaire)
        for client in arrivals:
            if client["type"] == "prio_abs":
                file.insert(0, client)  # priorité absolue en tête
            elif client["type"] == "prio_rel":
                # après les absolus mais avant les ordinaires
                idx = next((i for i, c in enumerate(file) if c["type"] == "ord"), len(file))
                file.insert(idx, client)
            else:
                file.append(client)
        
        if verbose:
            print("File (après placement):", file)
        
        # Service dans les stations
        for i in range(nbStations):
            if stations[i] is None and file:
                stations[i] = file.pop(0)
            
            if stations[i] is not None:
                stations[i]["duration"] -= 1
                if stations[i]["duration"] <= 0:
                    stations[i] = None
        
        file_cumulee += len(file)
        
        if verbose:
            print("Stations (fin):", stations)
            print("File (fin):", file)
    
    # Calcul du coût total
    cout_total = COUTS["inoccup"] * sum(1 for s in stations if s is None)
    cout_total += COUTS["presence_ord"] * file_cumulee
    return cout_total

# -------------------------------
# Exemple d’exécution
# -------------------------------
cout_min = simulate(nbStations=4, tempsSimul=20, verbose=True)
print("\nCoût total (4 stations):", cout_min)

# Pour les autres valeurs de stations
for s in range(5, 7):
    cout = simulate(nbStations=s, tempsSimul=20, verbose=False)
    print(f"Coût total ({s} stations):", cout)



=== Minute 1 ===
Stations (avant): [None, None, None, None]
File (avant): []
Arrivées: [{'type': 'ord', 'duration': 1}, {'type': 'ord', 'duration': 1}, {'type': 'ord', 'duration': 3}]
File (après placement): [{'type': 'ord', 'duration': 1}, {'type': 'ord', 'duration': 1}, {'type': 'ord', 'duration': 3}]
Stations (fin): [None, None, {'type': 'ord', 'duration': 2}, None]
File (fin): []

=== Minute 2 ===
Stations (avant): [None, None, {'type': 'ord', 'duration': 2}, None]
File (avant): []
Arrivées: [{'type': 'ord', 'duration': 1}, {'type': 'ord', 'duration': 1}, {'type': 'prio_rel', 'duration': 1}, {'type': 'prio_abs', 'duration': 2}]
File (après placement): [{'type': 'prio_abs', 'duration': 2}, {'type': 'prio_rel', 'duration': 1}, {'type': 'ord', 'duration': 1}, {'type': 'ord', 'duration': 1}]
Stations (fin): [{'type': 'prio_abs', 'duration': 1}, None, {'type': 'ord', 'duration': 1}, None]
File (fin): [{'type': 'ord', 'duration': 1}]

=== Minute 3 ===
Stations (avant): [{'type': 'prio_a

In [60]:
import numpy as np
import random

# ===============================
# PARAMÈTRES DU SYSTÈME
# ===============================

lambda_ordinaire = 1.5        # arrivées / minute
lambda_prioritaire = 0.7     # arrivées / minute
p_prioritaire_absolu = 0.30

temps_simulation = 60        # minutes
nbStationsMin = 2
nbStationsMax = 6

# coûts horaires → conversion en coût par minute
couts = {
    "attente_ordinaire": 15 / 60,
    "attente_prio_rel": 35 / 60,
    "attente_prio_abs": 45 / 60,
    "service_ordinaire": 28 / 60,
    "service_prio": 33 / 60,
    "station_vide": 18 / 60,
    "perte_prio": 20,
    "perte_ordinaire": 15
}

# Durées de service
durees_service = [1, 2, 3, 4, 5, 6]
frequences = [24, 18, 10, 3, 3, 2]
probas_service = np.array(frequences) / sum(frequences)

# ===============================
# GÉNÉRATION DES ARRIVÉES
# ===============================

def generer_arrivees():
    n_ord = np.random.poisson(lambda_ordinaire)
    n_prio = np.random.poisson(lambda_prioritaire)

    clients = []

    for _ in range(n_ord):
        clients.append(("ordinaire", False))

    for _ in range(n_prio):
        absolu = random.random() < p_prioritaire_absolu
        clients.append(("prioritaire", absolu))

    return clients

def generer_duree_service():
    return np.random.choice(durees_service, p=probas_service)

# ===============================
# SIMULATION POUR UN NOMBRE DE STATIONS
# ===============================

def simuler(nb_stations, afficher=False):
    stations = [None] * nb_stations
    files = {
        "prio_abs": [],
        "prio_rel": [],
        "ordinaire": []
    }

    cout_total = 0

    for minute in range(1, temps_simulation + 1):

        # 1️⃣ Arrivées
        arrivants = generer_arrivees()
        for c in arrivants:
            if c[0] == "prioritaire":
                if c[1]:
                    files["prio_abs"].append(c)
                else:
                    files["prio_rel"].append(c)
            else:
                files["ordinaire"].append(c)

        # 2️⃣ Traitement des stations
        for i in range(nb_stations):
            # station libre
            if stations[i] is None:
                client = None
                if files["prio_abs"]:
                    client = files["prio_abs"].pop(0)
                elif files["prio_rel"]:
                    client = files["prio_rel"].pop(0)
                elif files["ordinaire"]:
                    client = files["ordinaire"].pop(0)

                if client:
                    stations[i] = {
                        "client": client,
                        "temps": generer_duree_service()
                    }

            # station occupée
            if stations[i]:
                stations[i]["temps"] -= 1
                if stations[i]["temps"] <= 0:
                    stations[i] = None

        # 3️⃣ Coûts d’attente
        cout_total += (
            len(files["ordinaire"]) * couts["attente_ordinaire"]
            + len(files["prio_rel"]) * couts["attente_prio_rel"]
            + len(files["prio_abs"]) * couts["attente_prio_abs"]
        )

        # 4️⃣ Coûts stations
        for s in stations:
            if s is None:
                cout_total += couts["station_vide"]
            else:
                if s["client"][0] == "prioritaire":
                    cout_total += couts["service_prio"]
                else:
                    cout_total += couts["service_ordinaire"]

        # Affichage demandé pour le minimum de stations
        if afficher and minute <= 20:
            print(f"\nMinute {minute}")
            print("Stations :", stations)
            print("Files :", files)

    return cout_total

# ===============================
# RECHERCHE DU NOMBRE OPTIMAL
# ===============================

def recherche_nb_stations_optimal():
    resultats = {}

    for s in range(nbStationsMin, nbStationsMax + 1):
        cout = simuler(s, afficher=(s == nbStationsMin))
        resultats[s] = cout
        print(f"Stations = {s} → Coût total = {cout:.2f} €")

    optimal = min(resultats, key=resultats.get)
    print("\n✅ Nombre optimal de stations :", optimal)
    return optimal

# ===============================
# LANCEMENT
# ===============================

if __name__ == "__main__":
    recherche_nb_stations_optimal()



Minute 1
Stations : [None, None]
Files : {'prio_abs': [], 'prio_rel': [], 'ordinaire': []}

Minute 2
Stations : [{'client': ('ordinaire', False), 'temps': 2}, None]
Files : {'prio_abs': [], 'prio_rel': [], 'ordinaire': []}

Minute 3
Stations : [{'client': ('ordinaire', False), 'temps': 1}, {'client': ('ordinaire', False), 'temps': 5}]
Files : {'prio_abs': [], 'prio_rel': [], 'ordinaire': [('ordinaire', False)]}

Minute 4
Stations : [None, {'client': ('ordinaire', False), 'temps': 4}]
Files : {'prio_abs': [('prioritaire', True)], 'prio_rel': [('prioritaire', False), ('prioritaire', False), ('prioritaire', False)], 'ordinaire': [('ordinaire', False), ('ordinaire', False)]}

Minute 5
Stations : [{'client': ('prioritaire', True), 'temps': 1}, {'client': ('ordinaire', False), 'temps': 3}]
Files : {'prio_abs': [], 'prio_rel': [('prioritaire', False), ('prioritaire', False), ('prioritaire', False), ('prioritaire', False)], 'ordinaire': [('ordinaire', False), ('ordinaire', False)]}

Minute 6
